<a href="https://colab.research.google.com/github/Serajummunira/genz-social-media-analysis/blob/main/GenZ_Social_Media_Mental_Health_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📱 Gen-Z Social Media & Mental Health Analysis
### A Data Science Portfolio Project

---

**Author:** Serajum Munira  
**Course:** IFSC 77003-H02 - Data Science & Technologies  
**Supervisor:** Dr. Elizabeth Pierce  
**Semester:** Spring 2026  

---

## Project Overview

This notebook investigates **social media usage patterns and their association with mental health** among Generation Z individuals (ages 18–26). Using a synthetic dataset of 500 participants across 5 countries, we apply:

- 📊 **Exploratory Data Analysis (EDA)** — descriptive stats, distributions, group breakdowns
- 📈 **Inferential Statistics** — Pearson correlation, t-tests, one-way ANOVA
- 🎨 **Professional Visualizations** — 8 publication-quality figures

### Research Questions
| # | Research Question |
|---|---|
| RQ1 | Does higher daily usage correlate with lower mental health scores? |
| RQ2 | Does nighttime social media use predict worse mental health? |
| RQ3 | Do mental health scores differ by primary platform? |
| RQ4 | Are there gender and country-level differences in usage and wellbeing? |

---

> ⚠️ **Data Disclosure:** This dataset is **synthetic** (AI-generated) and does not represent real individuals. It is used here for academic demonstration of data science techniques. AI tools were also used to assist with code scaffolding — all analysis and interpretation is the author's own work.


---
## 📦 Section 1 — Setup & Imports


In [ ]:
# ── Core Libraries ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Global Plot Style ──────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0F1117',
    'axes.facecolor':   '#1A1D27',
    'axes.edgecolor':   '#2E3250',
    'axes.labelcolor':  '#E0E4FF',
    'axes.titlecolor':  '#FFFFFF',
    'axes.titlesize':   14,
    'axes.labelsize':   11,
    'xtick.color':      '#9099CC',
    'ytick.color':      '#9099CC',
    'grid.color':       '#2E3250',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'text.color':       '#E0E4FF',
    'font.family':      'DejaVu Sans',
    'figure.dpi':       120,
})

# ── Color Palette ──────────────────────────────────────────────────────────
ACCENT      = '#7C6AF7'          # Purple
ACCENT2     = '#F76A8C'          # Pink
ACCENT3     = '#6AF7C8'          # Teal
GENDER_PAL  = {'Female': '#F76A8C', 'Male': '#7C6AF7'}
NIGHT_PAL   = {'Yes': '#F76A8C',    'No':  '#6AF7C8'}
PLATFORM_COLORS = ['#7C6AF7', '#F76A8C', '#6AF7C8', '#F7C56A', '#F76A6A']
COUNTRY_COLORS  = ['#7C6AF7', '#F76A8C', '#6AF7C8', '#F7C56A', '#A8F76A']

print('✅ Libraries imported and plot style configured.')

---
## 📂 Section 2 — Load & Inspect the Dataset


In [ ]:
# ── Load Data ──────────────────────────────────────────────────────────────
# If running locally, adjust this path. In Colab, upload the CSV first.
# To upload in Colab: click the folder icon on the left → upload file

CSV_PATH = '/content/drive/MyDrive/genz_social_media_500_data.csv'   # ← change if needed

df = pd.read_csv(CSV_PATH)

print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'\nColumn names: {list(df.columns)}')
print('\nData types:')
print(df.dtypes)
print('\nFirst 5 rows:')
df.head()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Missing Values & Data Quality Check ───────────────────────────────────
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'No missing values found ✅')

print('\n=== Duplicate Rows ===')
dupes = df.duplicated().sum()
print(f'{dupes} duplicate rows found' if dupes > 0 else 'No duplicate rows ✅')

print('\n=== Value Ranges ===')
print(f'Age:                  {df.Age.min()} – {df.Age.max()}')
print(f'Daily_Usage_Hours:    {df.Daily_Usage_Hours.min()} – {df.Daily_Usage_Hours.max()}')
print(f'Num_Platforms_Used:   {df.Num_Platforms_Used.min()} – {df.Num_Platforms_Used.max()}')
print(f'Mental_Health_Score:  {df.Mental_Health_Score.min()} – {df.Mental_Health_Score.max()}')

---
## 📊 Section 3 — Exploratory Data Analysis (EDA)


In [ ]:
# ── Descriptive Statistics ─────────────────────────────────────────────────
print('=== Numeric Variables ===')
display(df[['Age','Daily_Usage_Hours','Num_Platforms_Used','Mental_Health_Score']]
        .describe().round(2))

print('\n=== Categorical Variables ===')
for col in ['Gender', 'Country', 'Primary_Platform', 'Purpose', 'Night_Usage']:
    counts = df[col].value_counts()
    pcts   = df[col].value_counts(normalize=True).mul(100).round(1)
    summary = pd.DataFrame({'Count': counts, 'Pct (%)': pcts})
    print(f'\n{col}:')
    display(summary)

In [ ]:
# ── Figure 1: Usage Hours Distribution ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0F1117')
fig.suptitle('Figure 1 — Daily Social Media Usage Distribution',
             fontsize=15, fontweight='bold', color='white', y=1.02)

# Histogram
ax = axes[0]
ax.hist(df['Daily_Usage_Hours'], bins=20, color=ACCENT, edgecolor='#0F1117', linewidth=0.8, alpha=0.85)
ax.axvline(df['Daily_Usage_Hours'].mean(), color=ACCENT2, linestyle='--', linewidth=2,
           label=f'Mean = {df["Daily_Usage_Hours"].mean():.2f} hrs')
ax.axvline(df['Daily_Usage_Hours'].median(), color=ACCENT3, linestyle=':', linewidth=2,
           label=f'Median = {df["Daily_Usage_Hours"].median():.2f} hrs')
ax.set_title('All Participants', fontweight='bold')
ax.set_xlabel('Hours per Day')
ax.set_ylabel('Frequency')
ax.legend(framealpha=0.3)
ax.grid(True, alpha=0.3)

# Boxplot by gender
ax2 = axes[1]
genders = df['Gender'].unique()
data_by_gender = [df[df['Gender']==g]['Daily_Usage_Hours'] for g in genders]
bp = ax2.boxplot(data_by_gender, patch_artist=True,
                  medianprops={'color': 'white', 'linewidth': 2})
colors_g = [GENDER_PAL[g] for g in genders]
for patch, color in zip(bp['boxes'], colors_g):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
ax2.set_xticks(range(1, len(genders)+1))
ax2.set_xticklabels(genders)
ax2.set_title('By Gender', fontweight='bold')
ax2.set_xlabel('Gender')
ax2.set_ylabel('Hours per Day')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig1_usage_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

# Interpretation
print('📌 Interpretation:')
print(f'  • Average daily usage: {df["Daily_Usage_Hours"].mean():.2f} hours')
print(f'  • Distribution is roughly uniform (1–8 hrs), suggesting diverse usage patterns')
print(f'  • Male vs female distributions appear similar in central tendency')

In [ ]:
# ── Figure 2: Mental Health Score Distribution ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0F1117')
fig.suptitle('Figure 2 — Mental Health Score Distribution',
             fontsize=15, fontweight='bold', color='white', y=1.02)

# Histogram
ax = axes[0]
ax.hist(df['Mental_Health_Score'], bins=8, color=ACCENT3, edgecolor='#0F1117', linewidth=0.8, alpha=0.85)
ax.axvline(df['Mental_Health_Score'].mean(), color=ACCENT2, linestyle='--', linewidth=2,
           label=f'Mean = {df["Mental_Health_Score"].mean():.2f}')
ax.axvline(df['Mental_Health_Score'].median(), color=ACCENT, linestyle=':', linewidth=2,
           label=f'Median = {df["Mental_Health_Score"].median():.2f}')
ax.set_title('All Participants', fontweight='bold')
ax.set_xlabel('Mental Health Score (3–10, higher = better)')
ax.set_ylabel('Frequency')
ax.legend(framealpha=0.3)
ax.grid(True, alpha=0.3)

# Boxplot by gender
ax2 = axes[1]
data_mh_gender = [df[df['Gender']==g]['Mental_Health_Score'] for g in genders]
bp2 = ax2.boxplot(data_mh_gender, patch_artist=True,
                   medianprops={'color': 'white', 'linewidth': 2})
for patch, color in zip(bp2['boxes'], colors_g):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
ax2.set_xticks(range(1, len(genders)+1))
ax2.set_xticklabels(genders)
ax2.set_title('By Gender', fontweight='bold')
ax2.set_xlabel('Gender')
ax2.set_ylabel('Mental Health Score')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig2_mental_health_dist.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

print('📌 Interpretation:')
print(f'  • Mean mental health score: {df["Mental_Health_Score"].mean():.2f} / 10')
print(f'  • Scores cluster around 5 and 8–9, suggesting bimodal distribution')
print(f'  • Gender differences are visually minimal')

In [ ]:
# ── Figure 3: Platform & Purpose Breakdown ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0F1117')
fig.suptitle('Figure 3 — Platform & Purpose Distribution',
             fontsize=15, fontweight='bold', color='white', y=1.02)

# Platform bar chart
platform_counts = df['Primary_Platform'].value_counts()
ax = axes[0]
bars = ax.bar(platform_counts.index, platform_counts.values,
              color=PLATFORM_COLORS, edgecolor='#0F1117', linewidth=0.8)
for bar, val in zip(bars, platform_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'n={val}', ha='center', va='bottom', fontsize=10, color='white')
ax.set_title('Primary Platform Used', fontweight='bold')
ax.set_xlabel('Platform')
ax.set_ylabel('Count')
ax.grid(True, alpha=0.3, axis='y')

# Purpose pie chart
purpose_counts = df['Purpose'].value_counts()
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    purpose_counts.values, labels=purpose_counts.index,
    colors=[ACCENT, ACCENT2, ACCENT3], autopct='%1.1f%%', startangle=140,
    textprops={'color': 'white'}, pctdistance=0.75,
    wedgeprops={'edgecolor': '#0F1117', 'linewidth': 2}
)
for at in autotexts:
    at.set_color('white')
    at.set_fontweight('bold')
ax2.set_title('Social Media Purpose', fontweight='bold')

plt.tight_layout()
plt.savefig('fig3_platform_purpose.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

print('📌 Interpretation:')
print('  • Twitter and Instagram are the most used platforms (~22% each)')
print('  • Purpose is split nearly evenly: Education (34%), Communication (34%), Entertainment (32%)')

---
## 🔬 Section 4 — Inferential Statistics


In [ ]:
# ── RQ1: Pearson Correlation — Usage Hours vs Mental Health ───────────────
print('=' * 60)
print('RQ1: Is daily usage negatively correlated with mental health?')
print('=' * 60)

r, p = stats.pearsonr(df['Daily_Usage_Hours'], df['Mental_Health_Score'])
n = len(df)

# Cohen's conventions for r: small=.10, medium=.30, large=.50
effect = 'Small' if abs(r) < 0.30 else ('Medium' if abs(r) < 0.50 else 'Large')

print(f'\n  Pearson r  = {r:.4f}')
print(f'  p-value    = {p:.6f}  {"< 0.001 ✅" if p < 0.001 else ""}')
print(f'  N          = {n}')
print(f'  Effect size: {effect}')
print(f'\n  R² = {r**2:.4f}  ({r**2*100:.1f}% of variance in MH explained by usage)')

print('\n📌 Conclusion:')
if p < 0.05:
    direction = 'negative' if r < 0 else 'positive'
    print(f'  Strong {direction} correlation found (r={r:.3f}, p<0.001).')
    print(f'  → Higher daily usage is significantly associated with lower mental health scores.')
else:
    print('  No significant correlation found.')

In [ ]:
# ── Figure 4: Scatter Plot — Usage vs Mental Health ───────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0F1117')

for gender, color in GENDER_PAL.items():
    subset = df[df['Gender'] == gender]
    ax.scatter(subset['Daily_Usage_Hours'], subset['Mental_Health_Score'],
               label=gender, alpha=0.55, color=color, s=55, edgecolors='none')

# Regression line
m, b = np.polyfit(df['Daily_Usage_Hours'], df['Mental_Health_Score'], 1)
x_range = np.linspace(df['Daily_Usage_Hours'].min(), df['Daily_Usage_Hours'].max(), 200)
ax.plot(x_range, m * x_range + b, color='white', linewidth=2.5, linestyle='--',
        label=f'OLS Trend  y = {m:.2f}x + {b:.2f}\nr = {r:.3f}, p < 0.001', zorder=5)

# Confidence interval shading
from scipy.stats import t as t_dist
x_mean = df['Daily_Usage_Hours'].mean()
se_line = np.sqrt(np.sum((df['Mental_Health_Score'] - (m * df['Daily_Usage_Hours'] + b))**2) / (n-2))
ci = t_dist.ppf(0.975, n-2) * se_line * np.sqrt(1/n + (x_range - x_mean)**2 / np.sum((df['Daily_Usage_Hours'] - x_mean)**2))
ax.fill_between(x_range, m*x_range+b-ci, m*x_range+b+ci, color='white', alpha=0.08, label='95% CI')

ax.set_title('Figure 4 — Daily Usage Hours vs. Mental Health Score', fontsize=14, fontweight='bold')
ax.set_xlabel('Daily Usage Hours')
ax.set_ylabel('Mental Health Score (higher = better)')
ax.legend(framealpha=0.25, fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig4_scatter_usage_mh.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

In [ ]:
# ── RQ2: Independent Samples t-Test — Night Usage vs Mental Health ────────
print('=' * 60)
print('RQ2: Does nighttime usage predict worse mental health?')
print('=' * 60)

night_yes = df[df['Night_Usage'] == 'Yes']['Mental_Health_Score']
night_no  = df[df['Night_Usage'] == 'No']['Mental_Health_Score']

t_stat, p_val = stats.ttest_ind(night_yes, night_no)
cohen_d = (night_yes.mean() - night_no.mean()) / np.sqrt(
    ((len(night_yes)-1)*night_yes.std()**2 + (len(night_no)-1)*night_no.std()**2) /
    (len(night_yes)+len(night_no)-2)
)

print(f'\n  Night Usage = Yes  →  Mean MH = {night_yes.mean():.3f}, SD = {night_yes.std():.3f}, n = {len(night_yes)}')
print(f'  Night Usage = No   →  Mean MH = {night_no.mean():.3f}, SD = {night_no.std():.3f}, n = {len(night_no)}')
print(f'\n  t-statistic = {t_stat:.4f}')
print(f'  p-value     = {p_val:.4f}  {"< 0.05 ✅" if p_val < 0.05 else "≥ 0.05 (not significant)"}')
print(f"  Cohen's d   = {cohen_d:.4f} (effect size)")

print('\n📌 Conclusion:')
if p_val < 0.05:
    print(f'  Significant difference found (p={p_val:.4f}). Night usage affects mental health.')
else:
    print(f'  No significant difference (p={p_val:.4f}). Night usage alone does not predict mental health.')
    print('  → The binary yes/no variable may not capture the full nuance (e.g., duration, content type).')

In [ ]:
# ── Figure 5: Night Usage vs Mental Health ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0F1117')
fig.suptitle('Figure 5 — Nighttime Usage and Mental Health',
             fontsize=15, fontweight='bold', color='white', y=1.02)

# Bar chart of means with error bars
ax = axes[0]
groups   = ['Yes (Night)', 'No (Night)']
means    = [night_yes.mean(), night_no.mean()]
errors   = [night_yes.sem(), night_no.sem()]
bar_cols = [NIGHT_PAL['Yes'], NIGHT_PAL['No']]
bars = ax.bar(groups, means, color=bar_cols, edgecolor='#0F1117', linewidth=0.8, alpha=0.85)
ax.errorbar(groups, means, yerr=errors, fmt='none', color='white', capsize=6, linewidth=2)
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.08,
            f'{val:.2f}', ha='center', fontweight='bold', color='white')
ax.set_ylim(0, 10)
ax.set_title('Mean Mental Health Score\n(error bars = SEM)', fontweight='bold')
ax.set_ylabel('Mean Mental Health Score')
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.98, 0.05, f'p = {p_val:.3f}', transform=ax.transAxes,
        ha='right', va='bottom', color='#9099CC', fontsize=10)

# Distribution KDE
ax2 = axes[1]
for label, data, color in [('Yes (Night)', night_yes, NIGHT_PAL['Yes']),
                             ('No (Night)',  night_no,  NIGHT_PAL['No'])]:
    data.plot.kde(ax=ax2, label=label, color=color, linewidth=2.5)
    ax2.fill_between(
        np.linspace(data.min(), data.max(), 200),
        stats.gaussian_kde(data)(np.linspace(data.min(), data.max(), 200)),
        alpha=0.2, color=color
    )
ax2.set_title('Score Distribution (KDE)', fontweight='bold')
ax2.set_xlabel('Mental Health Score')
ax2.set_ylabel('Density')
ax2.legend(framealpha=0.3)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig5_night_usage_mh.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

In [ ]:
# ── RQ3: One-Way ANOVA — Platform vs Mental Health ────────────────────────
print('=' * 60)
print('RQ3: Do mental health scores differ by primary platform?')
print('=' * 60)

platforms = df['Primary_Platform'].unique()
groups_platform = [df[df['Primary_Platform'] == p]['Mental_Health_Score'] for p in platforms]

f_stat, p_anova = stats.f_oneway(*groups_platform)

print('\n  Per-Platform Mean Mental Health Scores:')
platform_stats = df.groupby('Primary_Platform')['Mental_Health_Score'].agg(['mean','std','count'])
platform_stats.columns = ['Mean', 'SD', 'n']
platform_stats = platform_stats.sort_values('Mean', ascending=False)
display(platform_stats.round(3))

print(f'\n  One-Way ANOVA:')
print(f'  F-statistic = {f_stat:.4f}')
print(f'  p-value     = {p_anova:.4f}  {"< 0.05 ✅" if p_anova < 0.05 else "≥ 0.05 (not significant)"}')

print('\n📌 Conclusion:')
if p_anova < 0.05:
    print('  Significant differences exist across platforms.')
else:
    print(f'  No significant platform differences (p={p_anova:.3f}).')
    print('  → Platform choice alone does not differentiate mental health in this dataset.')

In [ ]:
# ── Figure 6: Platform vs Mental Health (Boxplot + Mean Dots) ─────────────
fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor('#0F1117')

platform_order = platform_stats.index.tolist()
color_map = dict(zip(platform_order, PLATFORM_COLORS))

data_by_platform = [df[df['Primary_Platform']==p]['Mental_Health_Score'] for p in platform_order]
bp = ax.boxplot(data_by_platform, patch_artist=True,
                medianprops={'color': 'white', 'linewidth': 2},
                whiskerprops={'color': '#9099CC'},
                capprops={'color': '#9099CC'},
                flierprops={'marker': 'o', 'markerfacecolor': '#9099CC', 'markersize': 4, 'alpha': 0.5})

for patch, plat in zip(bp['boxes'], platform_order):
    patch.set_facecolor(color_map[plat])
    patch.set_alpha(0.7)

# Overlay mean dots
for i, (plat, data) in enumerate(zip(platform_order, data_by_platform), 1):
    ax.scatter(i, data.mean(), color='white', s=80, zorder=5)
    ax.text(i, data.mean() + 0.2, f'{data.mean():.2f}', ha='center',
            fontsize=9, fontweight='bold', color='white')

ax.set_xticks(range(1, len(platform_order)+1))
ax.set_xticklabels(platform_order)
ax.set_title(f'Figure 6 — Mental Health Score by Primary Platform\n(ANOVA: F={f_stat:.3f}, p={p_anova:.3f})',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Primary Platform')
ax.set_ylabel('Mental Health Score')
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.98, 0.02, '● = group mean', transform=ax.transAxes,
        ha='right', va='bottom', color='#9099CC', fontsize=9)

plt.tight_layout()
plt.savefig('fig6_platform_mh.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

In [ ]:
# ── RQ4: Country & Gender Comparisons ─────────────────────────────────────
print('=' * 60)
print('RQ4: Country and gender differences in usage and mental health')
print('=' * 60)

print('\n--- By Country ---')
country_stats = df.groupby('Country').agg(
    Mean_MH=('Mental_Health_Score','mean'),
    Std_MH=('Mental_Health_Score','std'),
    Mean_Usage=('Daily_Usage_Hours','mean'),
    n=('Mental_Health_Score','count')
).round(3)
display(country_stats.sort_values('Mean_MH', ascending=False))

print('\n--- By Gender ---')
gender_stats = df.groupby('Gender').agg(
    Mean_MH=('Mental_Health_Score','mean'),
    Std_MH=('Mental_Health_Score','std'),
    Mean_Usage=('Daily_Usage_Hours','mean'),
    n=('Mental_Health_Score','count')
).round(3)
display(gender_stats)

# Gender t-test for mental health
female_mh = df[df['Gender']=='Female']['Mental_Health_Score']
male_mh   = df[df['Gender']=='Male']['Mental_Health_Score']
t_g, p_g = stats.ttest_ind(female_mh, male_mh)
print(f'\n  Gender t-test on Mental Health: t={t_g:.3f}, p={p_g:.4f}')
print(f'  {"Significant" if p_g < 0.05 else "No significant"} gender difference in mental health scores.')

In [ ]:
# ── Figure 7: Country Comparisons ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0F1117')
fig.suptitle('Figure 7 — Country-Level Comparisons',
             fontsize=15, fontweight='bold', color='white', y=1.02)

country_mh    = df.groupby('Country')['Mental_Health_Score'].mean().sort_values(ascending=False)
country_usage = df.groupby('Country')['Daily_Usage_Hours'].mean().sort_values(ascending=False)

for ax, data, title, ylabel, colors in [
    (axes[0], country_mh,    'Avg Mental Health Score by Country',  'Mean MH Score',     COUNTRY_COLORS),
    (axes[1], country_usage, 'Avg Daily Usage Hours by Country',    'Mean Usage (hours)', COUNTRY_COLORS)
]:
    bars = ax.bar(data.index, data.values, color=colors, edgecolor='#0F1117', linewidth=0.8, alpha=0.85)
    for bar, val in zip(bars, data.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{val:.2f}', ha='center', fontweight='bold', color='white', fontsize=10)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Country')
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('fig7_country_comparisons.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

print('📌 Interpretation:')
print('  • India shows the highest avg MH score; Australia the lowest.')
print('  • USA respondents report the highest average daily usage.')
print('  • These are descriptive; formal ANOVA would be needed to confirm significance.')

In [ ]:
# ── Figure 8: Correlation Heatmap ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor('#0F1117')

num_cols = ['Age', 'Daily_Usage_Hours', 'Num_Platforms_Used', 'Mental_Health_Score']
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle mask

cmap = sns.diverging_palette(240, 10, as_cmap=True)  # blue → red
sns.heatmap(corr, annot=True, fmt='.3f', cmap=cmap,
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5, linecolor='#0F1117',
            mask=mask,
            annot_kws={'size': 12, 'weight': 'bold'},
            cbar_kws={'shrink': 0.8})

ax.set_title('Figure 8 — Correlation Matrix (Numeric Variables)',
             fontsize=14, fontweight='bold', color='white', pad=15)
ax.tick_params(colors='white')
ax.set_facecolor('#1A1D27')

plt.tight_layout()
plt.savefig('fig8_correlation_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

print('📌 Key correlations:')
print(f'  • Daily_Usage_Hours ↔ Mental_Health_Score: r = {corr.loc["Daily_Usage_Hours","Mental_Health_Score"]:.3f}  ← strongest relationship')
print(f'  • All other numeric pairs: |r| < 0.06  (negligible)')

---
## 📋 Section 5 — Results Summary


In [ ]:
# ── Summary Results Table ──────────────────────────────────────────────────
results = pd.DataFrame({
    'Research Question': [
        'RQ1: Usage hours vs mental health',
        'RQ2: Night usage vs mental health',
        'RQ3: Platform vs mental health',
        'RQ4: Gender vs mental health'
    ],
    'Test': [
        'Pearson Correlation',
        'Independent t-test',
        'One-Way ANOVA',
        'Independent t-test'
    ],
    'Statistic': [
        f'r = {r:.3f}',
        f't = {t_stat:.3f}',
        f'F = {f_stat:.3f}',
        f't = {t_g:.3f}'
    ],
    'p-value': [
        f'{p:.4f}' if p >= 0.0001 else '< 0.0001',
        f'{p_val:.4f}',
        f'{p_anova:.4f}',
        f'{p_g:.4f}'
    ],
    'Significant?': [
        '✅ Yes' if p < 0.05 else '❌ No',
        '✅ Yes' if p_val < 0.05 else '❌ No',
        '✅ Yes' if p_anova < 0.05 else '❌ No',
        '✅ Yes' if p_g < 0.05 else '❌ No'
    ],
    'Key Finding': [
        'Strong negative correlation: more usage → lower MH',
        'No significant difference between night and non-night users',
        'No significant MH differences across platforms',
        'No significant gender differences in MH scores'
    ]
})

print('=== RESULTS SUMMARY TABLE ===')
display(results.set_index('Research Question'))

print('\n=== KEY TAKEAWAY ===')
print(f'  The only significant predictor of mental health in this dataset is')
print(f'  DAILY USAGE HOURS (r = {r:.3f}, p < 0.001, large effect).')
print(f'  Every additional hour of daily usage is associated with a {m:.3f} point')
print(f'  decrease in mental health score (out of 10).')

---
## 💾 Section 6 — Export All Figures


In [ ]:
# ── List all saved figures ─────────────────────────────────────────────────
import os
figs = sorted([f for f in os.listdir('.') if f.startswith('fig') and f.endswith('.png')])
print(f'Figures saved in current directory ({len(figs)} files):')
for f in figs:
    size_kb = os.path.getsize(f) / 1024
    print(f'  {f}  ({size_kb:.1f} KB)')

print('\n✅ All figures exported successfully!')
print('   → In Colab: use the Files panel (left sidebar) to download them.')
print('   → For GitHub: commit this .ipynb + figures folder to your repository.')

---

## 📚 References

- Twenge, J. M. et al. (2018). Increases in depressive symptoms... *Clinical Psychological Science*, 6(1), 3–17.
- Kelly, Y. et al. (2019). Social media use and adolescent mental health. *EClinicalMedicine*, 6, 59–68.
- Keles, B. et al. (2020). Systematic review: social media on depression/anxiety. *Int. J. Adolescence & Youth*, 25(1), 79–93.
- Orben, A. & Przybylski, A. K. (2019). Digital technology use and adolescent well-being. *Nature Human Behaviour*, 3(2), 173–182.
- Levenson, J. C. et al. (2017). Social media before bed and sleep disturbance. *Sleep*, 40(9).

---
*End of notebook — Serajum Munira · Spring 2026*
